# Import library, dataset

In [257]:
!pip install datasets

In [258]:
from datasets import load_dataset
import pandas as pd
import re
import string
import numpy as np

In [259]:
dataset = load_dataset("ruslanmv/ai-medical-chatbot")
train_data = dataset["train"]
df = pd.DataFrame(train_data)

print(df.head())
print(f"Dataset size: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

                                         Description  \
0      Q. What does abutment of the nerve root mean?   
1  Q. What should I do to reduce my weight gained...   
2  Q. I have started to get lots of acne on my fa...   
3  Q. Why do I have uncomfortable feeling between...   
4  Q. My symptoms after intercourse threatns me e...   

                                             Patient  \
0  Hi doctor,I am just wondering what is abutting...   
1  Hi doctor, I am a 22-year-old female who was d...   
2  Hi doctor! I used to have clear skin but since...   
3  Hello doctor,I am having an uncomfortable feel...   
4  Hello doctor,Before two years had sex with a c...   

                                              Doctor  
0  Hi. I have gone through your query with dilige...  
1  Hi. You have really done well with the hypothy...  
2  Hi there Acne has multifactorial etiology. Onl...  
3  Hello. The popping and discomfort what you fel...  
4  Hello. The HIV test uses a finger prick blood ..

# Deduplication, check NA, text cleaning

In [260]:
# Deduplication

# Check for duplicates before removal
num_total = len(df)
print(f"Total entries before deduplication: {num_total}")

# Identify duplicates (where all three fields are the same)
duplicate_mask = df.duplicated(subset=['Description', 'Patient', 'Doctor'], keep='first')
num_duplicates = duplicate_mask.sum()
print(f"Number of duplicate entries: {num_duplicates}")

# Remove duplicates, keeping only the first occurrence
df_clean = df.drop_duplicates(subset=['Description', 'Patient', 'Doctor'], keep='first')
print(f"Data shape after removing duplicates: {df_clean.shape}")
print(f"Removed {num_duplicates} duplicate entries")

# Basic statistics
print(df.isnull().sum()) # Check for missing values

Total entries before deduplication: 256916
Number of duplicate entries: 10378
Data shape after removing duplicates: (246538, 3)
Removed 10378 duplicate entries
Description    0
Patient        0
Doctor         0
dtype: int64


In [261]:
# Check for NA

print(df.isnull().sum()) # Check for missing values
# there is no missing values so no need to remove anything

Description    0
Patient        0
Doctor         0
dtype: int64


In [262]:
# Text cleaning and normalization

# Function to clean and normalize text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # Remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

# Apply cleaning to all text columns
df_clean['Description'] = df_clean['Description'].apply(clean_text)
df_clean['Patient'] = df_clean['Patient'].apply(clean_text)
df_clean['Doctor'] = df_clean['Doctor'].apply(clean_text)

# Change to lower case
df_clean['Description'] = df_clean['Description'].str.lower()
df_clean['Patient'] = df_clean['Patient'].str.lower()
df_clean['Doctor'] = df_clean['Doctor'].str.lower()

print(f"Text cleaning completed for {len(df_clean)} rows")

/tmp/ipython-input-3455620693.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Description'] = df_clean['Description'].apply(clean_text)
/tmp/ipython-input-3455620693.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Patient'] = df_clean['Patient'].apply(clean_text)
/tmp/ipython-input-3455620693.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

Text cleaning completed for 246538 rows


/tmp/ipython-input-3455620693.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Patient'] = df_clean['Patient'].str.lower()
/tmp/ipython-input-3455620693.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Doctor'] = df_clean['Doctor'].str.lower()


# Remove entries with very short questions or answers

In [263]:
min_desc_length = 10
min_doctor_length = 20

df_filtered = df_clean[
    (df_clean['Description'].str.len() >= min_desc_length) &
    (df_clean['Doctor'].str.len() >= min_doctor_length)
]

# Remove entries with extremely long text (to avoid exceeding model context window)
max_length = 1000

df_filtered = df_filtered[
    (df_filtered['Description'].str.len() <= max_length) &
    (df_filtered['Patient'].str.len() <= max_length) &
    (df_filtered['Doctor'].str.len() <= max_length)
]

print(f"Entries after length filtering: {len(df_filtered)} (removed {len(df_clean) - len(df_filtered)} rows)")

Entries after length filtering: 222110 (removed 24428 rows)


#Remove Q. from Description column

In [264]:
q_prefix_count = df_filtered['Description'].str.startswith('Q.').sum()

print(f"Descriptions starting with 'Q.': {q_prefix_count}")

# Function to remove the "Q." prefix and trim whitespace
def remove_q_prefix(text):
    if not isinstance(text, str):
        return ""
    if text.startswith('Q.'): # Remove "Q." prefix
        text = text[2:]
    return text.strip() # Trim leading whitespace

# Apply the function to the Description column
df_filtered['Description'] = df_filtered['Description'].apply(remove_q_prefix)

# Verify the changes
print("\nSample descriptions after removing 'Q.' prefix:\n")
for desc in df_clean['Description'].head(3):
    print(desc)
print("\n")

print(f"Entries after removing Q. prefix: {len(df_filtered)}")

Descriptions starting with 'Q.': 0

Sample descriptions after removing 'Q.' prefix:

q. what does abutment of the nerve root mean?
q. what should i do to reduce my weight gained due to genetic hypothyroidism?
q. i have started to get lots of acne on my face, particularly on my forehead. please help me.


Entries after removing Q. prefix: 222110


In [265]:
df_filtered.head()

,Description,Patient,Doctor
0,q. what does abutment of the nerve root mean?,"hi doctor,i am just wondering what is abutting...",hi. i have gone through your query with dilige...
2,q. i have started to get lots of acne on my fa...,hi doctor! i used to have clear skin but since...,hi there acne has multifactorial etiology. onl...
3,q. why do i have uncomfortable feeling between...,"hello doctor,i am having an uncomfortable feel...",hello. the popping and discomfort what you fel...
5,q. i had a surgery which ended up with some fa...,"hello doctor,i had an emergency surgery six mo...",hello. if you are saying it is already six mon...
6,q. kindly explain about beta strep.,"hello doctor, my culture from my gynecologist ...",hello. there are lots of bacteria and other or...


# Remove greeting, thanks, signature in Patient, Doctor Column

In [266]:
patient_greetings = [
    "my dear doctor", "hey dear doctor", "dear doctor", "hi doctor", "hello doctor", "hi", "hello",
]

doctor_greetings = [
    "hi there", "hi dear", "hello dear", "hi", "hello", "dear"
]

thanks_patterns = [
    "thank you so much", "thanks in advance", "many thanks",
    "thanks, doctor", "thanks doctor", "thanks for your help",
    "thank you for your time", "thanks a lot", "thanks", "thank you"
]

In [267]:
def remove_greetings(text, greetings_list):
    if not isinstance(text, str):
        return ""

    text_lower = text.lower() # Create lowercase version for matching

    # Check if text starts with any greeting pattern
    for greeting in greetings_list:
        if text_lower.startswith(greeting):
            text = text[len(greeting):] # Remove the same number of characters from the original text
            break

    text = text.strip() # Clean up any leading punctuation or whitespace
    while len(text) > 0 and text[0] in ',.:;!?- ': # Remove any leading punctuation that might be left
        text = text[1:].strip()
    return text

In [268]:
def remove_signature(text):
    if not isinstance(text, str) or len(text) < 10:
        return text
    check_length = min(70, len(text))
    tail = text[-check_length:].lower()
    signature_indicators = ["regards", "sincerely", "best", "dr.", "dr ", "doctor"]
    earliest_pos = len(tail)

    for indicator in signature_indicators:
        pos = tail.find(indicator)
        if pos >= 0 and pos < earliest_pos:
            earliest_pos = pos

    # If we found a signature indicator, remove everything from that point
    if earliest_pos < len(tail):
        cut_index = len(text) - check_length + earliest_pos
        return text[:cut_index].strip()

    return text

In [269]:
def remove_thanks(text, thanks_patterns):
    if not isinstance(text, str) or len(text) < 5:
        return text

    # Convert text to lowercase for matching
    text_lower = text.lower()

    # Check the last part of the text (use more characters to catch longer patterns)
    check_length = min(50, len(text))
    tail = text_lower[-check_length:]

    # Find the position of any thanks pattern
    earliest_pos = len(text)
    matched_pattern = None

    for pattern in thanks_patterns:
        pos = tail.find(pattern.lower())
        if pos >= 0:
            # Convert position in tail to position in original text
            actual_pos = len(text) - check_length + pos
            if actual_pos < earliest_pos:
                earliest_pos = actual_pos
                matched_pattern = pattern

    # If we found a pattern, remove everything from that point
    if matched_pattern:
        return text[:earliest_pos].strip()

    return text

In [270]:
# Step 1: Remove greetings
df_filtered['Patient'] = df_filtered['Patient'].apply(lambda x: remove_greetings(x, patient_greetings))
df_filtered['Doctor'] = df_filtered['Doctor'].apply(lambda x: remove_greetings(x, doctor_greetings))

# Step 2: Remove signature
df_filtered['Doctor'] = df_filtered['Doctor'].apply(remove_signature)

# Step 3: Remove thanks
df_filtered['Patient'] = df_filtered['Patient'].apply(lambda x: remove_thanks(x, thanks_patterns))
df_filtered['Doctor'] = df_filtered['Doctor'].apply(lambda x: remove_thanks(x, thanks_patterns))

In [271]:
df_filtered.head()

,Description,Patient,Doctor
0,q. what does abutment of the nerve root mean?,i am just wondering what is abutting and abutm...,i have gone through your query with diligence ...
2,q. i have started to get lots of acne on my fa...,i used to have clear skin but since i moved to...,acne has multifactorial etiology. only acne so...
3,q. why do i have uncomfortable feeling between...,i am having an uncomfortable feeling in betwee...,the popping and discomfort what you felt is ei...
5,q. i had a surgery which ended up with some fa...,i had an emergency surgery six months ago punc...,if you are saying it is already six months sin...
6,q. kindly explain about beta strep.,"my culture from my gynecologist came back, sho...",there are lots of bacteria and other organisms...


In [272]:
# Helper function

# Check if "doctor" appears in the first 10-15 characters of each cleaned text
def check_leading_doctor(text, chars=15):
    if not isinstance(text, str) or len(text) < 3:
        return False

    # Look only at the beginning portion of the text
    start_text = text[:chars].lower()
    return 'doctor' in start_text

# Find rows with "doctor" at the beginning
leading_doctor_mask = df_filtered['Patient'].apply(check_leading_doctor)
leading_doctor_count = leading_doctor_mask.sum()

print(f"Found {leading_doctor_count} rows with 'doctor' in the first 15 characters")

# Print examples
if leading_doctor_count > 0:
    # Check if we need additional greeting patterns
    print("\nYou may need to add these additional greeting patterns:")
    patterns_to_add = set()
    for text in df_filtered[leading_doctor_mask]['Patient']:
        # Get first 15 chars and lowercase
        start = text[:15].lower()
        if 'doctor' in start:
            # Extract the greeting including "doctor"
            for i in range(len(start)):
                if start[i:].startswith('doctor'):
                    patterns_to_add.add(start[:i+6])
                    break

    for pattern in patterns_to_add:
        print(f"- \"{pattern}\"")

Found 1533 rows with 'doctor' in the first 15 characters

You may need to add these additional greeting patterns:
- "too doctor"
- "what doctor"
- "1hi doctor"
- "dr, my doctor"
- "my va doctor"
- "helio doctor"
- "gud aft,doctor"
- "hye doctor"
- "dera doctor"
- "namsthe doctor"
- "i need a doctor"
- "i went 2doctor"
- "hej doctor"
- "yes doctor"
- "hai! doctor"
- "dear mr. doctor"
- "doc,my doctor"
- "can a doctor"
- "hlw doctor"
- "hhello doctor"
- "evening doctor"
- "helow doctor"
- "morning doctor"
- "best doctor"
- "thank u doctor"
- "hai doctor"
- "my dad s doctor"
- "gm,doctor"
- "skindoctor"
- "dear , doctor"
- "yes. my doctor"
- "hay doctor"
- "ello doctor"
- "gods day doctor"
- "so my doctor"
- "hellow doctor"
- "q : helo doctor"
- "i am doctor"
- "why doctor"
- "doctor"
- "which doctor"
- "good doctor"
- "hallow doctor"
- "hy doctor"
- "hello doctor"
- "my sons doctor"
- "actually doctor"
- "my eye doctor"
- "thanks, doctor"
- "mygi doctor"
- "doc doctor"
- "sir, my doctor"

In [273]:
# Before applying thanks removal
print("\nChecking for thanks patterns:")
thanks_count = 0
for i in range(min(10000, len(df_filtered))):
    text = df_filtered['Patient'].iloc[i].lower()
    if any(pattern in text[-50:] for pattern in thanks_patterns):
        thanks_count += 1
        print(f"Found potential thanks in row {i}: ...{text[-50:]}")
print(f"Found {thanks_count} potential thanks patterns in first 20 rows")

# Make copies to check for changes
df_filtered['Patient_Before'] = df_filtered['Patient'].copy()
df_filtered['Doctor_Before'] = df_filtered['Doctor'].copy()

# Apply thanks removal
df_filtered['Patient'] = df_filtered['Patient'].apply(lambda x: remove_thanks(x, thanks_patterns))
df_filtered['Doctor'] = df_filtered['Doctor'].apply(lambda x: remove_thanks(x, thanks_patterns))

# Count modified entries
patient_modified = (df_filtered['Patient'] != df_filtered['Patient_Before']).sum()
doctor_modified = (df_filtered['Doctor'] != df_filtered['Doctor_Before']).sum()

print(f"Thanks removal modified {patient_modified} patient entries ({patient_modified/len(df_filtered):.2%})")
print(f"Thanks removal modified {doctor_modified} doctor entries ({doctor_modified/len(df_filtered):.2%})")


Checking for thanks patterns:
Found 0 potential thanks patterns in first 20 rows
Thanks removal modified 17 patient entries (0.01%)
Thanks removal modified 6 doctor entries (0.00%)


# Create QA Input

In [274]:
# Combine Description and Patient fields as specified in your proposal
df_filtered['Question'] = df_filtered.apply(
    lambda row: f"Medical Question: {row['Description']}\nPatient Information: {row['Patient']}",
    axis=1
)

# Use Doctor field as the answer
df_filtered['Answer'] = df_filtered['Doctor']

# Display a sample
print("Sample question-answer pair:")
sample_idx = df_filtered.index[0]
print(f"Question:\n{df_filtered.loc[sample_idx, 'Question'][:300]}...\n")
print(f"Answer:\n{df_filtered.loc[sample_idx, 'Answer'][:300]}...")

Sample question-answer pair:
Question:
Medical Question: q. what does abutment of the nerve root mean?
Patient Information: i am just wondering what is abutting and abutment of the nerve root means in a back issue. please explain. what treatment is required for annular bulging and tear?...

Answer:
i have gone through your query with diligence and would like you to know that i am here to help you. for further information consult a neurologist online -->...


In [275]:
# Create formatted input with instruction for T5
df_filtered['Formatted_Input'] = "Answer this medical question: " + df_filtered['Question']

# Keep relevant columns
df_final = df_filtered[['Formatted_Input', 'Question', 'Answer']]

In [276]:
df_final.head()

,Formatted_Input,Question,Answer
0,Answer this medical question: Medical Question...,Medical Question: q. what does abutment of the...,i have gone through your query with diligence ...
2,Answer this medical question: Medical Question...,Medical Question: q. i have started to get lot...,acne has multifactorial etiology. only acne so...
3,Answer this medical question: Medical Question...,Medical Question: q. why do i have uncomfortab...,the popping and discomfort what you felt is ei...
5,Answer this medical question: Medical Question...,Medical Question: q. i had a surgery which end...,if you are saying it is already six months sin...
6,Answer this medical question: Medical Question...,Medical Question: q. kindly explain about beta...,there are lots of bacteria and other organisms...


# Train Test Split

In [277]:
from sklearn.model_selection import train_test_split

# First split: 90% train+val, 10% test
train_val_df, test_df = train_test_split(df_final, test_size=0.1, random_state=42)

# Second split: split the 90% into ~89% train, ~11% val (resulting in 80/10/10 overall)
train_df, val_df = train_test_split(train_val_df, test_size=0.11, random_state=42)

# Verify split sizes
print(f"Total dataset: {len(df_final)} entries")
print(f"Training set: {len(train_df)} entries ({len(train_df)/len(df_final):.2%})")
print(f"Validation set: {len(val_df)} entries ({len(val_df)/len(df_final):.2%})")
print(f"Test set: {len(test_df)} entries ({len(test_df)/len(df_final):.2%})")

Total dataset: 222110 entries
Training set: 177910 entries (80.10%)
Validation set: 21989 entries (9.90%)
Test set: 22211 entries (10.00%)


In [278]:
# Save to CSV files
train_df.to_csv('medical_qa_train.csv', index=False)
val_df.to_csv('medical_qa_val.csv', index=False)
test_df.to_csv('medical_qa_test.csv', index=False)

print("Datasets saved as CSV files")

Datasets saved as CSV files
